# ArqSim quickstart: one circuit, two architectures

Run a small Clifford+T circuit through the public Python API and read the resulting cost estimates:

**`FTCircuit` + `EvaluationConfig` → `run_evaluation()` → `EvaluationReport`**

The saved outputs were generated with ArqSim 0.2.0. Every number below comes from an executed report. The code needs only the base ArqSim installation; it uses no frontend, server, plotting package, or optional synthesis backend.

## Run this notebook

From the repository root, in an activated Python environment:

```bash
python -m pip install .
python -m pip install jupyterlab
python -m jupyterlab examples/notebooks/quickstart.ipynb
```

Select that environment's Python kernel and run the cells in order. JupyterLab is only the notebook runner, not an ArqSim runtime dependency. An existing notebook runner with ArqSim installed in its kernel works too. Install ArqSim from this repository: the PyPI package with the bare name `arqsim` is unrelated. No cell installs packages or depends on the current working directory.


## 1. Load an FT-level circuit

Our toy input is `H(q0) → S(q0) → CX(q0, q1) → T(q1)`: two logical qubits and four ordered gates. It is the same circuit used in the repository's README and Python quickstart. It is already in the Clifford+T vocabulary, so it needs no external synthesis pass.

`load_ft_workload()` reads synthesized OpenQASM 2 and returns the canonical `FTCircuit`. For this self-contained example, we write the short QASM text to a temporary file and remove it immediately after loading. For your own input, pass your QASM file's path directly to the loader. Logical layers encode dependencies, not execution times.


In [1]:
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

from arqsim import EvaluationConfig, EvaluationReport, FTCircuit, run_evaluation
from arqsim.program import load_ft_workload

qasm = """OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
h q[0];
s q[0];
cx q[0],q[1];
t q[1];
"""
with TemporaryDirectory() as directory:
    source = Path(directory) / "quickstart.qasm"
    source.write_text(qasm, encoding="utf-8")
    circuit = load_ft_workload(source, representation="gate")

assert isinstance(circuit, FTCircuit)
print(f"ArqSim {version('arqsim')}")
print(f"Circuit: {circuit.num_qubits} logical qubits, {circuit.operation_count} gates")


ArqSim 0.2.0
Circuit: 2 logical qubits, 4 gates


## 2. Evaluate and read the summary

Profile **2.3** combines neutral-atom memory and compute with a remote superconducting magic-state factory. `run_evaluation()` resolves the reference architecture, compiles and lowers the circuit, runs the runtime model, and returns a checked report.

This example keeps the one-shot defaults: full tracing, the monolithic `black_box` T model, `canonical_reference_v1` fidelity, and `protocol_aware_reference_v1` footprint. In this mode, T is not expanded into measurement/reaction/correction children. Fidelity is enabled by default; only an explicit `fidelity_profile=None` disables it. Missing required fidelity coverage fails closed.


In [2]:
config = EvaluationConfig(profile_id="2.3", run_label="notebook-quickstart-2.3")
report = run_evaluation(circuit, config)
summary = report.summary

assert summary.fidelity_complete_coverage
assert summary.all_invariants_satisfied
print(f"Simulated latency: {summary.total_latency_s:.6f} s ({summary.total_latency_s * 1000:.3f} ms)")
print(f"Physical qubits: {summary.total_physical_qubits:,.0f}")
print(f"Success probability: {summary.success_probability:.8f}")
print(f"Complete fidelity coverage: {summary.fidelity_complete_coverage}")
print(f"Completed Program instructions: {summary.completed_program_instructions}")
print(f"Completed runtime events: {summary.event_count}")
print(f"All terminal invariants satisfied: {summary.all_invariants_satisfied}")


Simulated latency: 0.040291 s (40.291 ms)
Physical qubits: 6,286
Success probability: 0.99991204
Complete fidelity coverage: True
Completed Program instructions: 8
Completed runtime events: 12
All terminal invariants satisfied: True


| Summary field | How to read it |
| --- | --- |
| `total_latency_s` | Simulated end-to-end Program latency in **seconds**; multiply by 1,000 for milliseconds. It is not the notebook's wall-clock running time. |
| `total_physical_qubits` | Resolved architecture footprint under the selected footprint model; it is not the circuit's two logical qubits. |
| `success_probability` | Estimated success probability under the selected fidelity model. |
| `fidelity_complete_coverage` | Whether that fidelity estimate covers all required effects in the run. |
| `completed_program_instructions` | Completed lowered Program instructions, which need not equal the input gate count. |
| `event_count` | Completed runtime events, including Program work and resource-production work. |
| `all_invariants_satisfied` | Whether every named terminal runtime invariant passed. The individual checks are in `summary.invariant_checks`. |

These are system-level reference-model estimates, not measured hardware performance or a simulation of the algorithm's output quantum state. Complete coverage and passing invariants check the modeled run; they do not establish that its physical assumptions are calibrated for a particular device.


## 3. Change the architecture, keep the circuit

Profile **1.1** is a homogeneous neutral-atom architecture with colocated compute and magic-state supply. Compare it with Profile **2.3** using the same circuit and default model selections. Only the architecture selection and descriptive run label change.

Each profile resolves its own sizing, layout, QEC and resource-protocol bindings. This small comparison illustrates how to ask an architectural question; it is not an equal-hardware-budget experiment or a general ranking of platforms. Inspect the resolved inputs and use representative workloads before drawing a scientific conclusion.


In [3]:
reference_report = run_evaluation(
    circuit,
    EvaluationConfig(profile_id="1.1", run_label="notebook-quickstart-1.1"),
)
reports = {"1.1": reference_report, "2.3": report}
print(f"{'Profile':<8} {'Latency (ms)':>14} {'Physical qubits':>17} {'Success p':>13} {'Coverage':>10}")
for profile_id, result in reports.items():
    s = result.summary
    assert s.fidelity_complete_coverage and s.all_invariants_satisfied
    print(f"{profile_id:<8} {s.total_latency_s * 1000:>14.3f} {s.total_physical_qubits:>17,.0f} {s.success_probability:>13.8f} {str(s.fidelity_complete_coverage):>10}")


Profile    Latency (ms)   Physical qubits     Success p   Coverage
1.1            4073.404             1,811    0.99997825       True
2.3              40.291             6,286    0.99991204       True


## 4. Keep the evidence with the result

The report contains the request, resolved model inputs, compilation, execution plan, trace, and checked results. Its JSON is the portable evidence artifact. Here we round-trip it in memory without printing the large document or creating a file.

The strict loader validates the stored report and reconstructs its typed views; it does not rerun the compiler or runtime. The stable `summary` remains available after loading.


In [4]:
report_json = report.to_json()
loaded_report = EvaluationReport.from_json(report_json)
assert loaded_report.report_hash == report.report_hash
assert loaded_report.summary.to_dict() == report.summary.to_dict()
print("Report-v2 round trip verified: report identity and summary preserved.")


Report-v2 round trip verified: report identity and summary preserved.


## Next steps

- Use the [Python quickstart script](../quickstart.py) to load a small QASM file.
- Read the [public API guide](../../docs/01-public-api/public-api.md) for configuration, defaults, errors, and report access.
- Explore [Report and Trace](../../docs/01-public-api/report-and-trace-schema.md) when you need runtime evidence beyond the summary.
- See the [compiler IR walkthrough](../../docs/02-offline-pipeline/compiler-ir-walkthrough.md) for the lower-level compilation and plan boundaries.
